In [1]:
!pip install -q -U "gliner2[local]" transformers huggingface_hub seqeval scikit-learn matplotlib seaborn pandas tqdm torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.3 MB/s eta 0:00:00
   ━━━

In [2]:
import json
import random
from collections import Counter

custom_data = []
seen_texts = set()

# Using corrected_output_v2.jsonl since the v3 processing was aborted
with open('corrected_output_v3.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        text = ' '.join(item['tokenized_text'])
        if text not in seen_texts:
            seen_texts.add(text)
            custom_data.append(item)

print(f"Loaded {len(custom_data)} unique examples.")

random.seed(42)
random.shuffle(custom_data)
custom_data = custom_data[:8000]

print(f"Using {len(custom_data)} rows for training.")

labels = [e[2] for item in custom_data for e in item['ner']]
print("Label distribution:", Counter(labels))

Loaded 4470 unique examples.
Using 4470 rows for training.
Label distribution: Counter({'GPE': 6830, 'ORG': 6795, 'EVENT': 4508, 'PERSON': 1972, 'DATE': 1297})


In [3]:
train_ratio, val_ratio = 0.80, 0.10

train_idx = int(train_ratio * len(custom_data))
val_idx = train_idx + int(val_ratio * len(custom_data))

train_data = custom_data[:train_idx]
val_data = custom_data[train_idx:val_idx]
test_data = custom_data[val_idx:]

# Verify no data leakage
train_texts = set(' '.join(d['tokenized_text']) for d in train_data)
val_texts = set(' '.join(d['tokenized_text']) for d in val_data)
test_texts = set(' '.join(d['tokenized_text']) for d in test_data)

assert len(train_texts & val_texts) == 0, "LEAKAGE: train/val overlap!"
assert len(train_texts & test_texts) == 0, "LEAKAGE: train/test overlap!"
assert len(val_texts & test_texts) == 0, "LEAKAGE: val/test overlap!"

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")
print("No data leakage detected.")

Train: 3576, Val: 447, Test: 447
No data leakage detected.


In [ ]:
import torch
import gc
from pathlib import Path
from collections import defaultdict
from gliner2 import GLiNER2
from gliner2.training.data import InputExample, TrainingDataset
from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig

TARGET_LABELS = ['PERSON', 'ORG', 'GPE', 'DATE', 'EVENT']

ENTITY_DESCRIPTIONS = {
    "PERSON": "Names of people, including first names, last names, and full names",
    "ORG": "Names of organizations, companies, agencies, institutions",
    "GPE": "Names of countries, cities, states, and other geopolitical entities",
    "DATE": "Absolute or relative dates or periods of time",
    "EVENT": "Named events such as wars, battles, elections, protests, natural disasters, summits",
}

def convert_to_input_examples(dataset):
    examples = []
    skipped = 0
    for item in dataset:
        tokens = item['tokenized_text']
        text = ' '.join(tokens)
        ner = item['ner']

        if not ner:
            skipped += 1
            continue

        entities_by_label = defaultdict(list)
        for start_idx, end_idx, label in ner:
            if label in TARGET_LABELS:
                mention = ' '.join(tokens[start_idx:end_idx + 1])
                if mention.strip():
                    entities_by_label[label].append(mention)

        entities_dict = {label: list(set(entities_by_label.get(label, []))) for label in TARGET_LABELS}

        if not any(entities_dict.values()):
            skipped += 1
            continue

        examples.append(InputExample(
            text=text,
            entities=entities_dict,
            entity_descriptions=ENTITY_DESCRIPTIONS
        ))

    print(f"Converted {len(examples)} examples, skipped {skipped} empty")
    return examples

print("Converting train data...")
train_examples = convert_to_input_examples(train_data)
print("Converting val data...")
val_examples = convert_to_input_examples(val_data)
print("Converting test data...")
test_examples = convert_to_input_examples(test_data)

print("\nValidating training dataset...")
train_ds = TrainingDataset(train_examples)
train_ds.validate(raise_on_error=True)
print("Training data validated OK")
train_ds.print_stats()

# -- Load model --
BASE_MODEL = "fastino/gliner2-base-v1"
print(f"\nLoading base model: {BASE_MODEL}")
model = GLiNER2.from_pretrained(BASE_MODEL)

WORK_DIR = Path("./gliner2_finetuned")
WORK_DIR.mkdir(exist_ok=True)

config = TrainingConfig(
    output_dir=str(WORK_DIR / 'checkpoints'),
    experiment_name='geopolitical_ner_unified',
    num_epochs=8,
    batch_size=18,
    eval_batch_size=8,
    gradient_accumulation_steps=2,
    encoder_lr=1e-5,
    task_lr=5e-4,
    weight_decay=0.01,
    max_grad_norm=1.0,
    scheduler_type='cosine',
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    eval_strategy='epoch',
    save_best=True,
    save_total_limit=3,
    metric_for_best='eval_loss',
    greater_is_better=False,
    early_stopping=True,
    early_stopping_patience=3,
    early_stopping_threshold=0.0,
    use_lora=True,
    lora_r=8,
    lora_alpha=16.0,
    lora_dropout=0.05,
    lora_target_modules=['encoder'],
    save_adapter_only=True,
    logging_steps=50,
    logging_first_step=True,
    num_workers=2,
    pin_memory=True,
    seed=42
)

print("Training configuration created.")
print("Starting Fine-Tuning...")
trainer = GLiNER2Trainer(model=model, config=config)
train_result = trainer.train(train_data=train_examples, eval_data=val_examples)
print("Training completed!")

gc.collect()
torch.cuda.empty_cache()

Converting train data...
Converted 3415 examples, skipped 161 empty
Converting val data...
Converted 425 examples, skipped 22 empty
Converting test data...
Converted 430 examples, skipped 17 empty

Validating training dataset...
Training data validated OK

GLiNER2 Training Dataset Statistics
Total examples: 3415

Text lengths: min=34, max=673, mean=209.9

Task Distribution:
  entities_only: 3415 (100.0%)

Entity Types (16997 total mentions):
  ORG: 5390
  GPE: 5386
  EVENT: 3557
  PERSON: 1615
  DATE: 1049


Loading base model: fastino/gliner2-base-v1


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/823 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Evaluating:   0%|          | 0/54 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/54 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7cf7df6ba480>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7cf7df6ba480>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Evaluating:   0%|          | 0/54 [00:00<?, ?it/s]

Training completed!


In [5]:
import torch
from tqdm import tqdm
from collections import defaultdict
import pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

model = trainer.model
model.eval()

stats = {label: {"tp": 0, "fp": 0, "fn": 0} for label in TARGET_LABELS}

for item in tqdm(test_data, desc="Evaluating test set"):
    tokens = item['tokenized_text']
    text = ' '.join(tokens)
    ner = item['ner']

    # Gold entities as set of (mention, label)
    gold = set()
    for start_idx, end_idx, label in ner:
        if label in TARGET_LABELS:
            mention = ' '.join(tokens[start_idx:end_idx + 1]).strip().lower()
            gold.add((mention, label))

    # Predict
    with torch.no_grad():
        result = model.extract_entities(text, ENTITY_DESCRIPTIONS, include_confidence=True, include_spans=True)

    entities = result.get("entities", {}) if isinstance(result, dict) else result

    pred = set()
    if isinstance(entities, dict):
        for label, ent_list in entities.items():
            label = str(label).strip().upper()
            if label in TARGET_LABELS:
                for e in ent_list:
                    mention = e.get("text", e.get("mention", "")).strip().lower() if isinstance(e, dict) else str(e).strip().lower()
                    if mention:
                        pred.add((mention, label))

    # Score
    for label in TARGET_LABELS:
        gold_l = {m for m, l in gold if l == label}
        pred_l = {m for m, l in pred if l == label}
        stats[label]["tp"] += len(gold_l & pred_l)
        stats[label]["fp"] += len(pred_l - gold_l)
        stats[label]["fn"] += len(gold_l - pred_l)

# -- Compute metrics --
rows = []
total_tp = total_fp = total_fn = 0
for label in TARGET_LABELS:
    tp, fp, fn = stats[label]["tp"], stats[label]["fp"], stats[label]["fn"]
    total_tp += tp; total_fp += fp; total_fn += fn
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    rows.append({"Label": label, "Precision": f"{p:.4f}", "Recall (Sensitivity)": f"{r:.4f}", "F1": f"{f1:.4f}", "Support": tp + fn})

# Micro averages
micro_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
micro_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0
rows.append({"Label": "MICRO AVG", "Precision": f"{micro_p:.4f}", "Recall (Sensitivity)": f"{micro_r:.4f}", "F1": f"{micro_f1:.4f}", "Support": total_tp + total_fn})

df = pd.DataFrame(rows)
print("\n" + "=" * 70)
print("DETAILED EVALUATION ON TEST SET")
print("=" * 70)
print(df.to_string(index=False))
print(f"\nMicro F1: {micro_f1:.4f} | Micro Precision: {micro_p:.4f} | Micro Recall (Sensitivity): {micro_r:.4f}")

Evaluating test set: 100%|██████████| 447/447 [00:22<00:00, 20.00it/s]


DETAILED EVALUATION ON TEST SET
    Label Precision Recall (Sensitivity)     F1  Support
   PERSON    0.8785               0.9578 0.9164      166
      ORG    0.8416               0.9278 0.8826      693
      GPE    0.8811               0.9139 0.8972      697
     DATE    0.6410               0.8333 0.7246      120
    EVENT    0.6098               0.7750 0.6826      480
MICRO AVG    0.7851               0.8864 0.8327     2156

Micro F1: 0.8327 | Micro Precision: 0.7851 | Micro Recall (Sensitivity): 0.8864


In [7]:
# ── Test the Fine-Tuned Model on Custom Examples ──

test_sentences = [
    "An F-16 was shot down by Iraq during the Gulf War.",
    "Iranian general Qasem Soleimani was killed in a US drone airstrike near Baghdad airport on January 3, 2020."
]

# Make sure the model is in evaluation mode
model = trainer.model
model.eval()

print("==================================================")
print("🔍 TESTING CUSTOM EXAMPLES")
print("==================================================\n")

for i, text in enumerate(test_sentences):
    print(f"📝 Text {i+1}: {text}")

    # Extract entities
    with torch.no_grad():
        results = model.extract_entities(text, ENTITY_DESCRIPTIONS, include_confidence=True)

    # The output is wrapped in an "entities" key
    extracted_entities = results.get("entities", results)

    # Format and print the output nicely
    found_any = False
    for label, ent_list in extracted_entities.items():
        if ent_list:
            found_any = True
            print(f"  🟢 {label}:")
            for ent in ent_list:
                # Handle varying dictionary keys
                mention = ent.get('text', ent.get('mention', 'UNKNOWN'))
                conf = ent.get('confidence', ent.get('score', 0.0))
                print(f"      - {mention} (Confidence: {conf:.2f})")

    if not found_any:
        print("  ❌ No entities detected.")

    print("\n" + "-"*50 + "\n")

🔍 TESTING CUSTOM EXAMPLES

📝 Text 1: An F-16 was shot down by Iraq during the Gulf War.
  🟢 GPE:
      - Iraq (Confidence: 1.00)
  🟢 EVENT:
      - shot down (Confidence: 0.98)
      - Gulf War (Confidence: 0.71)

--------------------------------------------------

📝 Text 2: Iranian general Qasem Soleimani was killed in a US drone airstrike near Baghdad airport on January 3, 2020.
  🟢 PERSON:
      - Qasem Soleimani (Confidence: 1.00)
  🟢 GPE:
      - Baghdad airport (Confidence: 0.81)
  🟢 DATE:
      - January 3, 2020 (Confidence: 0.97)
  🟢 EVENT:
      - killed (Confidence: 0.97)
      - airstrike (Confidence: 0.93)

--------------------------------------------------

